In [1]:
import os
import sys
import time
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import CIFAR10
from torchvision import transforms
import torch.optim as optim

import torch.utils.data as data
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import matplotlib
matplotlib.rcParams['lines.linewidth'] = 2.0

In [2]:
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.7 MB/s eta 0:00:00


In [3]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

In [4]:
import torch
data_path = "data_path"
checkpoint_path = "checkpoint_path"

os.makedirs(checkpoint_path, exist_ok = True)

def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
set_seed(42)

torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0")if torch.cuda.is_available()else torch.device("cpu")
print("using this device", device)

using this device cuda:0


In [5]:
import urllib.error as HTTPError
import urllib.request
base_url = "https://raw.githubusercontent.com/phlippe/saved_models/main/"
pretrained_filename = ["tutorial15/ViT.ckpt", "tutorial15/tensorboards/ViT/events.out.tfevents.ViT","tutorial5/tensorboards/ResNet/events.out.tfevents.resnet"]

os.makedirs(checkpoint_path, exist_ok = True)
def pretrained_file(base_url:str, pretrained_filename:str):
  for file_name in pretrained_filename:
    file_url = os.path.join(checkpoint_path, file_name)
    if "/" in file_name:
      os.makedirs(os.path.dirname(file_url), exist_ok = True)
      if not os.path.isfile(file_url):
        file_path = base_url + file_name
        print(f"downloading this file_path {file_path}")
        try:
          urllib.request.urlretrieve(file_path, file_url)
        except HTTPError as e:
          print(f"downloading this file_name from google_drive/n", e)



if __name__ == "__main__":
  pretrained_file(base_url, pretrained_filename)

downloading this file_path https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial15/ViT.ckpt
downloading this file_path https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial15/tensorboards/ViT/events.out.tfevents.ViT
downloading this file_path https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial5/tensorboards/ResNet/events.out.tfevents.resnet


In [6]:
train_dataset = CIFAR10(root = data_path, train = True, download = True)
train_dataset

100%|██████████| 170M/170M [35:25<00:00, 80.2kB/s]


Dataset CIFAR10
    Number of datapoints: 50000
    Root location: data_path
    Split: Train

In [7]:
train_mean = (train_dataset.data/255.0).mean(axis = (0,1,2))
train_std = (train_dataset.data/255.0).std(axis = (0,1,2))
print(train_mean)
print(train_std)

[0.49139968 0.48215841 0.44653091]
[0.24703223 0.24348513 0.26158784]


In [8]:
train_transform = transforms.Compose([transforms.ToTensor(), transforms.RandomHorizontalFlip(p = 0.5),
                                      transforms.RandomResizedCrop((32, 32), scale = (0.8, 1.0), ratio = (0.8, 1.0)),
                                      transforms.Normalize(train_mean, train_std)])
test_transform = transforms.Compose([transforms.ToTensor(),
                                     transforms.Normalize(train_mean, train_std)])


train_datasets = CIFAR10(root = data_path, train = True, download = True, transform = train_transform)
val_datasets = CIFAR10(root = data_path, train = True, download = True, transform = test_transform)
test_datasets = CIFAR10(root = data_path, train = False, download = True, transform = test_transform)


train_full_dataset = len(train_datasets)
train_size = int(0.8 * train_full_dataset)
val_size = train_full_dataset - train_size
train_dataset = torch.utils.data.random_split(train_datasets, [train_size, val_size])
val_dataset = torch.utils.data.random_split(val_datasets, [train_size, val_size])


train_loader = data.DataLoader(train_datasets, batch_size = 256, shuffle = True, num_workers = 0, pin_memory_device = True)
val_loader = data.DataLoader(val_datasets, batch_size = 256, shuffle = False, num_workers = 0, pin_memory_device = True)
test_loader = data.DataLoader(test_datasets, batch_size = 256, shuffle = False, num_workers = 0, pin_memory_device = True)

print(train_loader)
print(val_loader)
print(test_loader)

In [9]:
def patch_embedding(x, patch_size, return_flatten = True):
  B, C, H, W = x.shape
  x = x.reshape(B, C, H//patch_size, patch_size, W//patch_size, patch_size)
  x = x.permute(0, 2, 4, 1, 3, 5)
  x = x.flatten(1,2)
  if return_flatten:
    x = x.flatten(2,4)
  return x

In [10]:
class Attention_Block(nn.Module):
  def __init__(self, embed_dim, num_head, hidden_dim, drop_out = 0.0):
    super().__init__()

    self.layer_norm_1 = nn.LayerNorm(embed_dim)
    self.multi_head = nn.MultiheadAttention(embed_dim, num_head, dropout = drop_out, batch_first=False) # Added batch_first for clarity
    self.layer_norm_2 = nn.LayerNorm(embed_dim)
    self.linear = nn.Sequential(nn.Linear(embed_dim, hidden_dim),
                                nn.GELU(),
                                nn.Dropout(drop_out),
                                nn.Linear(hidden_dim, embed_dim),
                                nn.Dropout(drop_out))



  def forward(self, x):
    # Corrected: Pass x as query, key, and value for self-attention
    x = x + self.multi_head(x, x, x)[0] # MultiheadAttention returns (attn_output, attn_output_weights)
    x = self.layer_norm_1(x)
    x = x + self.linear(x)
    x = self.layer_norm_2(x)
    return x

In [11]:
class CoreVisionTransformer(nn.Module):
  def __init__(self, embed_dim, num_head, hidden_dim, num_classes, num_channels, num_layers, num_patches,  patch_size, drop_out = 0.0):
    super().__init__()

    self.patch_size = patch_size
    self.input_layer = nn.Sequential(nn.Linear(num_channels * (patch_size **2),embed_dim))
    self.transformer = nn.Sequential(*[Attention_Block(embed_dim, num_head, hidden_dim, drop_out) for _ in range(num_layers)])
    self.mlp_head = nn.Sequential(nn.LayerNorm(embed_dim),
                                  nn.Linear(embed_dim, num_classes))
    self.drop_out = nn.Dropout(drop_out)



    ###parameters_founding####
    self.cls_token_parameter = nn.Parameter(torch.randn(1,1,embed_dim)) # Renamed to avoid conflict
    self.pos_embedding = nn.Parameter(torch.randn(1, 1+num_patches, embed_dim))



  def forward(self, x):
    x = patch_embedding(x, self.patch_size)
    B, T, _ = x.shape
    x = self.input_layer(x)

    cls_token = self.cls_token_parameter.repeat(B, 1, 1) # Updated reference
    x = torch.cat([cls_token, x], dim = 1)
    pos_embed = self.pos_embedding[:, :T+1]
    x = pos_embed + x


    x = self.drop_out(x)
    x = x.transpose(0, 1) # Corrected: call transpose on x, and specified dims (batch, sequence)
    x = self.transformer(x)

    cls = x[0]
    x = self.mlp_head(cls)
    return x



In [12]:
class Vision_Transformer(pl.LightningModule):
  def __init__(self, model_kwargs, lr):
    super().__init__()
    self.save_hyperparameters()
    self.model = CoreVisionTransformer(**model_kwargs)
    self.example_input_array = next(iter(train_loader))[0]



  def forward(self, x):
    x = self.model(x)
    return x # Added missing return statement


  def configure_optimizers(self):
    optimizer = optim.AdamW(self.parameters(), lr = self.hparams.lr)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones = [100, 150], gamma = 0.1)
    return [optimizer], [scheduler]



  def _calculate_loss(self, batch, mode = "train"):
    label, target = batch
    pred = self.model(label)
    loss = F.cross_entropy(pred, target)
    acc = (pred.argmax(dim = -1) == target).float().mean()

    self.log(f"{mode}_loss", loss)
    self.log(f"{mode}_acc", acc)


  def training_step(self, batch, batch_idx):
    loss  = self._calculate_loss(batch, mode = "train")
    return loss


  def validation_step(self, batch, batch_idx):
    self._calculate_loss(batch, mode = "val")

  def test_step(self, batch, batch_idx):
    self._calculate_loss(batch, mode = "test")

In [15]:
def train_model(**kwargs):
  root_dir = os.path.join(checkpoint_path, "VIT")

  trainer = pl.Trainer(default_root_dir = root_dir,
                       accelerator = "auto",
                       devices = 1,
                       max_epochs = 100,
                       min_epochs = 10,
                       callbacks = [ModelCheckpoint(save_weights_only = True, mode = "max", monitor = "val_acc"), LearningRateMonitor("epoch")])

  trainer.logger._log_graph = True
  trainer.logger._default_hp_metric = None


  pretrained_filename = os.path.join(checkpoint_path, "VIT.ckpt")
  # Corrected logic: if the file exists, load it. Otherwise, train a new model.
  if os.path.isfile(pretrained_filename):
    model = Vision_Transformer.load_from_checkpoint(pretrained_filename)
  else:
    pl.seed_everything(42)
    model = Vision_Transformer(**kwargs)
    trainer.fit(model, train_loader, val_loader)
    model = Vision_Transformer.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

  train_results = trainer.test(model, train_loader, verbose = False)
  test_results = trainer.test(model, test_loader, verbose = False)
  val_results = trainer.validate(model, val_loader, verbose = False)

  # Corrected keys for train_acc and val_acc
  results = {"train_acc":train_results[0]["test_acc"], "val_acc":val_results[0]["val_acc"], "test_acc":test_results[0]["test_acc"]}

  model = model.to(device)
  return model, results

In [16]:
trained_model, train_results = train_model(model_kwargs= {
    "embed_dim":128,
    "hidden_dim":512,
     "num_classes":10,
    "num_head":8,
    "num_layers":12,  # Increased from 6 to 12
    "num_patches":64,
    "patch_size":4,
    "num_channels":3,
    "drop_out":0.1},
      lr=5e-5) # Changed from 1e-4 to 5e-5

print("VIT_results", train_results)

Validation ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196/196 0:00:14 • 0:00:00 13.77it/s

VIT_results {'train_acc': 0.0938199982047081, 'val_acc': 0.09331999719142914, 'test_acc': 0.09319999814033508}
